# Sections 1 and 2: Positioning and Direct Replication

ECON 5200 Midterm Project · Replication Study

> **🛑 AI policy, and it is not the problem sets' policy.** Sections 1 and 2 — the
> positioning and the direct replication — are **no AI**. Not for the code, not for
> the prose. Making a computer agree with a published table is the exercise that
> establishes you can read a paper, and a model that has already read the paper will
> simply hand you the answer without your ever finding the discrepancy. Sections 3 to
> 6 authorise AI as a documented co-pilot under **P.R.I.M.E.**, on the same terms as
> every problem set: you file the full transcript in `ai-appendix.md`, and if you
> cannot explain a line you cannot submit it. State in your README which sections you
> used it in. This follows the course rule — manual command first, expansion second —
> and using AI in Sections 1 or 2 is an integrity violation, not a style disagreement.

## Setup

Imports and paths only.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

import statsmodels.api as sm
from linearmodels.iv import IV2SLS

# The repository root, whether this runs from notebooks/ (Jupyter) or from the root (reproduce.py).
ROOT = Path.cwd() if (Path.cwd() / "download_data.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
(ROOT / "output").mkdir(exist_ok=True)

from download_data import MY_PAPER, PAPERS, fetch

## Data

In [ ]:
paths = fetch(MY_PAPER)
card = pd.read_stata(paths[0])

print("Number of men:", len(card))

## Section 1: Positioning (12 pts)

Two pages maximum in the report. What question the paper asks, what was believed before it, what it changed, and how it has been received, with at least three citations to work that engages with it critically. Then the identification strategy and the assumption it rests on, in your own words.

The prose goes in the report. Keep your notes and the three citations here.

**Critical citations (at least three):**

1. Kling (2001)
2. Carneiro and Heckman (2002)
3. Cameron and Taber (2004)


**The identification strategy and its assumption, in my own words:**

Card uses proximity to a four-year college as an instrument for schooling to estimate how extra education affects wages, assuming that, after accounting for the controls, living near a college affects wages only through schooling and is unrelated to other unmeasured wage advantages.

## Section 2: Direct Replication (24 pts)

Reproduce the paper's headline result and report your number beside theirs. You will not match exactly. Document every discrepancy and its likely source: sample construction, variable definition, software defaults, a decision the paper does not describe. Where the paper is ambiguous about a choice, say so: those ambiguities are your Section 3.

In [ ]:
CONTROLS = [
    "exper", "expersq", "black", "south", "smsa", "smsa66",
    "reg662", "reg663", "reg664", "reg665",
    "reg666", "reg667", "reg668", "reg669"
]

wages = card["lwage"]
schooling = card["educ"]
near_college = card["nearc4"]

iv_controls = sm.add_constant(card[CONTROLS])
iv_fit = IV2SLS(wages, iv_controls, schooling, near_college).fit(
    cov_type="unadjusted", debiased=True
)

ols_controls = sm.add_constant(card[["educ"] + CONTROLS])
ols_fit = sm.OLS(wages, ols_controls).fit()

In [ ]:
if iv_fit is None or ols_fit is None:
    print("Fill Section 2 first: iv_fit and ols_fit are not set yet.")
else:
    fs = sm.OLS(card["educ"], sm.add_constant(card[["nearc4"] + CONTROLS])).fit()
    print(f"{'':<8}{'return to a year':>18}{'SE':>9}")
    print(f"{'OLS':<8}{ols_fit.params['educ']:>18.4f}{ols_fit.bse['educ']:>9.4f}")
    print(f"{'2SLS':<8}{iv_fit.params['educ']:>18.4f}{iv_fit.std_errors['educ']:>9.4f}")
    print(f"first stage on nearc4: {fs.params['nearc4']:+.3f} years, F = {fs.tvalues['nearc4'] ** 2:.1f}")

### Theirs vs mine

One row per number you replicate. The row label says where the number is in the paper (for example, `Table 2, col 3: effect on log wage`). `gap_pct` is computed for you as 100 × (mine − theirs) / |theirs|. This cell writes `output/headline_table.csv`, which `python reproduce.py` prints and your README quotes.

In [ ]:
headline = table.copy()

headline["gap_pct"] = (
    100
    * (headline["replication"] - headline["paper"])
    / headline["paper"].abs()
)

headline.to_csv(ROOT / "output" / "headline_table.csv")

print(headline.round(4).to_string())

### Discrepancy log

One row per discrepancy, including the ones you could not explain. A clean match reported without investigation is less credible than a 3% gap that is explained.

| # | Estimate | What differs | Size of the gap | Likely source | How I checked it | Is the paper ambiguous here? |
|---|---|---|---|---|---|---|
| 1 | Schooling, 2SLS | 0.1315 vs 0.132 | −0.0005 | Rounding | Checked the instrument, controls and sample against Table 3A, column 5; the coefficient and SE match after rounding. | No apparent ambiguity in the specification. |
| 2 | College proximity on schooling | 0.3199 vs 0.320 | −0.0001 | Rounding | Checked the schooling outcome, 1966 proximity indicator and controls against Table 3A, column 1. | No apparent ambiguity. |
| 3 | College proximity on log wages | 0.0421 vs 0.042 | +0.0001 | Rounding | Checked the 1976 log-wage outcome and controls against Table 3A, column 3; dividing the unrounded wage effect by the schooling effect reproduces 2SLS. | No apparent ambiguity. |
| 4 | Schooling, OLS | 0.0747 vs 0.075; SE 0.0035 vs 0.003 | −0.0003 | Rounding | Checked Table 2, column 2; the full SE of about 0.003498 rounds directly to 0.003 at three decimals. | No apparent ambiguity in the specification. |
| 5 | Experience | 0.0848 vs 0.085 | −0.0002 | Rounding | Checked that experience squared is also included; the coefficient and SE match after rounding. | No apparent ambiguity. |
| 6 | Experience squared, scaled | −0.2287 vs −0.229 | +0.0003 | Scaling and rounding | Checked the row label and multiplied the coefficient and SE by 100 to match the paper’s scale. | No; the row label gives the scale. |
| 7 | Black indicator | −0.1990 vs −0.199 | −0.0000 | Rounding | Checked Table 2, column 2; the difference is slightly negative but rounds to zero at four decimals. | No apparent ambiguity. |
| 8 | Southern residence in 1976 | −0.1480 vs −0.148 | 0.0000 | Rounding | Checked that the indicator refers to 1976 residence; the coefficient and SE match at the paper’s precision. | No; the table notes identify the year. |
| 9 | Urban residence in 1976 | 0.1364 vs 0.136 | +0.0004 | Rounding | Checked that smsa refers to 1976 and smsa66 is included separately as a control. | No; the table notes distinguish the years. |
| 10 | OLS R-squared | 0.2998 vs 0.300 | −0.0002 | Rounding | Checked the R-squared from the OLS model matching Table 2, column 2. | No apparent ambiguity. |
| 11 | Sample size | 3,010 vs 3,010 | 0 | Matching sample | Checked the fitted-model observation counts against the paper’s table notes. | No; the paper reports 3,010 men. |

**Ambiguities in the paper, to take into Section 3:**

My baseline results match the paper once rounded, so there aren’t any unexplained gaps in the numbers. In Section 3, I check how well the finding holds up when I change the standard errors, split the sample by childhood location, allow college proximity to affect wages beyond schooling, and look at family background.

In [ ]:
table.to_csv(ROOT / "output" / "headline_table.csv")